# Olist E-Commerce Analysis
## Notebook 4 — Customer Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA = 'C:/Users/loayi/OneDrive/Desktop/Workspace/olist-analysis/data/'

df = pd.read_csv(DATA + 'master.csv')
reviews = pd.read_csv(DATA + 'olist_order_reviews_dataset.csv')
translations = pd.read_csv(DATA + 'product_category_name_translation.csv')
orders = pd.read_csv(DATA + 'olist_orders_dataset.csv')

df = pd.merge(df, translations, on='product_category_name', how='left')
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
df = df.drop(columns=['order_purchase_timestamp'], errors='ignore')
df = pd.merge(df, orders[['order_id', 'order_purchase_timestamp']], on='order_id', how='left')

print(df.shape)
print('loaded')

## 1. Top 10 Cities by Number of Customers

In [ ]:
dick = df['customer_city'].value_counts().head(10)
plt.figure(figsize=(12, 6))
plt.barh(dick.index, dick.values)
plt.title('Top 10 Cities by Number of Customers')
plt.xlabel('Number of Customers')
plt.ylabel('City')
plt.tight_layout()
plt.show()

**Insight:** São Paulo has the highest customer concentration by a large margin, consistent with it being Brazil's largest city and economic center.

## 2. Repeat Buyers vs One-Time Buyers

In [ ]:
counts = df['customer_unique_id'].value_counts()
repeat = (counts > 1).sum()
one_time = (counts == 1).sum()

print(f'One-time buyers: {one_time}')
print(f'Repeat buyers: {repeat}')

d = {'One-time': one_time, 'Repeat': repeat}
plt.pie(d.values(), labels=d.keys(), autopct='%1.1f%%')
plt.title('One-time vs Repeat Buyers')
plt.show()

**Insight:** 87.5% of customers only purchased once, indicating a major retention problem. Olist's business is almost entirely dependent on acquiring new customers. A loyalty program or targeted re-engagement campaign could significantly improve lifetime value.

## 3. Review Score Distribution by State

In [ ]:
merged = pd.merge(reviews, df, how='inner', on='order_id')
plt.figure(figsize=(16, 6))
sns.boxplot(data=merged, x='customer_state', y='review_score')
plt.title('Review Score Distribution by State')
plt.xlabel('State')
plt.ylabel('Review Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight:** Roraima (RR) has the lowest median review score, likely due to longer delivery times to this remote northern state — a hypothesis to verify in the delivery analysis notebook.

## 4. Average Review Score by Product Category

In [ ]:
dick = merged.groupby('product_category_name_english')['review_score'].mean().sort_values()
plt.figure(figsize=(14, 20))
plt.barh(dick.index, dick.values)
plt.title('Average Review Score by Product Category')
plt.xlabel('Average Score')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

**Insight:** Security & Services has the lowest average review score, suggesting product quality or delivery issues in this category. Olist should investigate seller performance in this segment.

## 5. Monthly New Customers

In [ ]:
first_order = df.groupby('customer_unique_id')['order_purchase_timestamp'].min()
first_order_month = first_order.dt.to_period('M')
new_customers = first_order_month.value_counts().sort_index()

new_customers.plot(figsize=(12, 5), marker='o')
plt.title('Monthly New Customers')
plt.xlabel('Month')
plt.ylabel('New Customers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight:** New customer acquisition grew steadily through 2017 and into 2018, peaking around November 2017 (Black Friday). Combined with the low repeat buyer rate, this confirms Olist is heavily acquisition-driven.